In [7]:
import sys
from pathlib import Path

import pandas as pd


# Paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "assignment2" / "data" / "db"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))

from src.features import (
    build_activity_events,
    build_features,
    build_labels,
)


# Configuration
WINDOWS = [7, 30, 90]
TARGET_HORIZON_DAYS = 7

DATA_START = pd.Timestamp(
    "2025-12-01 00:00:00",
    tz="UTC"
)

FINAL_PREDICTION_TIME = pd.Timestamp(
    "2026-11-22 00:00:00",
    tz="Asia/Taipei"
).tz_convert("UTC")

In [8]:
table_names = [
    "Users",
    "Repositories",
    "PullRequests",
    "Issues",
    "IssueComments",
    "PRReviews",
    "PRReviewComments",
    "Pushes",
]

tables = {}

for name in table_names:
    path = DATA_DIR / f"{name}.parquet"

    tables[name] = pd.read_parquet(path)

users = tables["Users"]
repos = tables["Repositories"]
prs = tables["PullRequests"]
issues = tables["Issues"]
issue_comments = tables["IssueComments"]
pr_reviews = tables["PRReviews"]
pr_review_comments = tables["PRReviewComments"]
pushes = tables["Pushes"]

In [9]:
activity_events = build_activity_events(
    prs=prs,
    issues=issues,
    issue_comments=issue_comments,
    pr_reviews=pr_reviews,
    pr_review_comments=pr_review_comments,
    pushes=pushes
)

In [10]:
LATEST_ROW_TIME = activity_events["first_seen_at"].max()

if pd.isna(LATEST_ROW_TIME):
    raise ValueError(
        "No valid row times were found in activity_events."
    )

DATA_END = LATEST_ROW_TIME.floor("D")

print("Latest row time:", LATEST_ROW_TIME)
print("Data end:", DATA_END)


FIRST_TRAIN_CUTOFF = (
    DATA_START
    + pd.Timedelta(days=max(WINDOWS))
)

VAL_CUTOFF = (
    DATA_END
    - pd.Timedelta(days=TARGET_HORIZON_DAYS)
)

LAST_TRAIN_CUTOFF = (
    VAL_CUTOFF
    - pd.Timedelta(days=TARGET_HORIZON_DAYS)
)

TRAIN_CUTOFFS = pd.date_range(
    start=FIRST_TRAIN_CUTOFF,
    end=LAST_TRAIN_CUTOFF,
    freq="7D"
)

Latest row time: 2026-11-09 08:59:58+00:00
Data end: 2026-11-09 00:00:00+00:00


In [11]:
training_snapshots = []

for cutoff in TRAIN_CUTOFFS:
    print("Building snapshot:", cutoff)

    snapshot_features = build_features(
        users=users,
        activity_events=activity_events,
        prediction_time=cutoff,
        windows=WINDOWS
    )

    snapshot_labels = build_labels(
        users=users,
        activity_events=activity_events,
        prediction_time=cutoff,
        horizon_days=TARGET_HORIZON_DAYS
    )

    snapshot = snapshot_features.merge(
        snapshot_labels,
        on="user_id",
        how="left",
        validate="one_to_one"
    )

    training_snapshots.append(snapshot)

training_df = pd.concat(
    training_snapshots,
    ignore_index=True
)

training_path = (
    OUTPUT_DIR / "training_features.parquet"
)

training_df.to_parquet(
    training_path,
    index=False
)

Building snapshot: 2026-03-01 00:00:00+00:00
Building snapshot: 2026-03-08 00:00:00+00:00
Building snapshot: 2026-03-15 00:00:00+00:00
Building snapshot: 2026-03-22 00:00:00+00:00
Building snapshot: 2026-03-29 00:00:00+00:00
Building snapshot: 2026-04-05 00:00:00+00:00
Building snapshot: 2026-04-12 00:00:00+00:00
Building snapshot: 2026-04-19 00:00:00+00:00
Building snapshot: 2026-04-26 00:00:00+00:00
Building snapshot: 2026-05-03 00:00:00+00:00
Building snapshot: 2026-05-10 00:00:00+00:00
Building snapshot: 2026-05-17 00:00:00+00:00
Building snapshot: 2026-05-24 00:00:00+00:00
Building snapshot: 2026-05-31 00:00:00+00:00
Building snapshot: 2026-06-07 00:00:00+00:00
Building snapshot: 2026-06-14 00:00:00+00:00
Building snapshot: 2026-06-21 00:00:00+00:00
Building snapshot: 2026-06-28 00:00:00+00:00
Building snapshot: 2026-07-05 00:00:00+00:00
Building snapshot: 2026-07-12 00:00:00+00:00
Building snapshot: 2026-07-19 00:00:00+00:00
Building snapshot: 2026-07-26 00:00:00+00:00
Building s

In [12]:
val_features = build_features(
    users=users,
    activity_events=activity_events,
    prediction_time=VAL_CUTOFF,
    windows=WINDOWS
)

val_labels = build_labels(
    users=users,
    activity_events=activity_events,
    prediction_time=VAL_CUTOFF,
    horizon_days=TARGET_HORIZON_DAYS
)

validation_df = val_features.merge(
    val_labels,
    on="user_id",
    how="left",
    validate="one_to_one"
)

validation_path = (
    OUTPUT_DIR / "validation_features.parquet"
)

validation_df.to_parquet(
    validation_path,
    index=False
)

print("Validation data shape:", validation_df.shape)
print("Saved:", validation_path)

Validation data shape: (9000, 42)
Saved: /Users/triciachew/BigData2026/outputs/validation_features.parquet
